In [0]:
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.functions import *

from dataclasses import dataclass

In [0]:
%sql
create or replace table session_11_framework.config_table(
    pipeline_name string
  , file_path string
  , header string
  , delimiter string
  , table_name string
  , schema_detail map<string,string>
  , keys array<string>
  , write_mode string
)

In [0]:
def upsert_into(df:DataFrame,table_name:str,keys:list) -> DataFrame:
    delta_obj = DeltaTable.forName(spark,table_name)
    return (
        delta_obj.alias("t").merge(
            df.alias("s")," AND ".join([f"t.{key} = s.{key}" for key in keys])
            )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
        )

In [0]:
data = [{
    "pipeline_name": "shop_name",
    "file_path": "/Volumes/session_11_streaming/default/source_csv",
    "header": "true",
    "delimiter": "|",
    "table_name": "session_11_framework.shop_name",
    "schema_detail": {"shop_id": "int", "shop_name": "string", "branch_name": "string", "file_dt": "date"},
    "keys": ["shop_id"],
    "write_mode": "overwrite"
}]

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("header", StringType(), True),
    StructField("delimiter", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("schema_detail", MapType(StringType(), StringType()), True),
    StructField("keys", ArrayType(StringType()), True),
    StructField("write_mode", StringType(), True)
])

mock_df = spark.createDataFrame(data, schema)
mock_df.display()
upsert_into(mock_df,"session_11_framework.config_table",["pipeline_name"])


In [0]:
spark.table("session_11_framework.config_table").display()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, MapType, ArrayType

data = [{
    "pipeline_name": "fact_sales",
    "file_path": "/Volumes/session_11_streaming/default/source_parquet",
    "header": None,
    "delimiter":None,
    "table_name": "session_11_framework.fact_sales",
    "schema_detail": {"transaction_id": "int","shop_id": "int","sales_qty": "int","sales_amt": "string","sales_date": "date"},
    "keys": ["transaction_id","shop_id"],
    "write_mode": "overwrite"
}]

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("header", StringType(), True),
    StructField("delimiter", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("schema_detail", MapType(StringType(), StringType()), True),
    StructField("keys", ArrayType(StringType()), True),
    StructField("write_mode", StringType(), True)
])

mock_df = spark.createDataFrame(data, schema)
mock_df = spark.createDataFrame(data, schema)
mock_df.display()
upsert_into(mock_df,"session_11_framework.config_table",["pipeline_name"])

In [0]:
spark.table("session_11_framework.config_table").display()